<a href="https://colab.research.google.com/github/isaacrich080-ctrl/Google-Landmark-Recognition/blob/main/Google_Landmark_Recognition.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
import os
import gc
import sys
import time
import cv2
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

import tensorflow as tf
import tensorflow.keras as keras
from tensorflow.keras import layers
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from tensorflow.keras import regularizers

# Fix for compatibility with newer TensorFlow versions if experimental preprocessing is deprecated
try:
    from tensorflow.keras.layers.experimental import preprocessing
except ImportError:
    preprocessing = tf.keras.layers

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix

SEED = 42


In [ ]:
from google.colab import files

uploaded = files.upload()

In [ ]:
import pandas as pd

train_df = pd.read_csv("train.csv")

print(train_df.head())
print(train_df.shape)

In [ ]:
import os
import numpy as np
import pandas as pd

def load_traindf():
    # Directly reads train.csv uploaded in Colab session storage
    csv_path = 'train.csv'

    traindf = pd.read_csv(csv_path)

    # If you also uploaded images to a folder named 'train' inside Colab
    traindf['img_path'] = traindf['id'].apply(
        lambda r: os.path.join('train', r[0], r[1], r[2], f"{r}.jpg")
    )

    traindf['landmark_id'] = traindf['landmark_id'].apply(lambda x: np.int32(x))
    return traindf

# Run data loading
traindf = load_traindf()
landmark_unique = len(traindf['landmark_id'].unique())
print(f"Loaded {len(traindf):,} rows with {landmark_unique:,} unique landmarks.")
# Reading and resizing images from their path
def img_read_resize(img_path):
    img = plt.imread(img_path)
    img_redim = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
    return img_redim


def get_size(obj, seen=None):
    """Recursively finds size of objects"""
    size = sys.getsizeof(obj)
    if seen is None:
        seen = set()
    obj_id = id(obj)
    if obj_id in seen:
        return 0
    # Important mark as seen *before* entering recursion to gracefully handle
    # self-referential objects
    seen.add(obj_id)
    if isinstance(obj, dict):
        size += sum([get_size(v, seen) for v in obj.values()])
        size += sum([get_size(k, seen) for k in obj.keys()])
    elif hasattr(obj, '__dict__'):
        size += get_size(obj.__dict__, seen)
    elif hasattr(obj, '__iter__') and not isinstance(obj, (str, bytes, bytearray)):
        size += sum([get_size(i, seen) for i in obj])
    return size


traindf = load_traindf()
landmark_unique = len(traindf['landmark_id'].unique())
traindf



In [ ]:
get_size(traindf)

In [ ]:
traindf.info(memory_usage='deep')

In [ ]:
print('Data from the original training dataset \n')
print('Number of images in the dataset: ', traindf.shape[0])
print('Number of different classes: ', landmark_unique)
print('Min class: ', min(traindf['landmark_id']))
print('Max class: ', max(traindf['landmark_id']))
print('\nRepetitions of elements by class:')
print(traindf['landmark_id'].value_counts())

In [ ]:
import os
import matplotlib.pyplot as plt
import numpy as np

folder = "/content/landmark_small"

files = [
    os.path.join(folder, f)
    for f in os.listdir(folder)
    if f.endswith(".jpg")
]

print("Images found:", len(files))

plt.figure(figsize=(15, 10))

for i in range(min(20, len(files))):
    j = np.random.randint(0, len(files))

    img = plt.imread(files[j])

    plt.subplot(4, 5, i + 1)
    plt.imshow(img)
    plt.xticks([])
    plt.yticks([])

plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
import os

traindf = pd.DataFrame({
    'id': [os.path.basename(f).split('_label')[0] for f in os.listdir('/content/landmark_small') if f.endswith('.jpg')],
    'landmark_id': [int(os.path.basename(f).split('_label')[1].replace('.jpg','')) for f in os.listdir('/content/landmark_small') if f.endswith('.jpg')],
    'img_path': [os.path.join('/content/landmark_small', f) for f in os.listdir('/content/landmark_small') if f.endswith('.jpg')]
})

print(traindf.head())
print("Total images:", len(traindf))
print("Total classes:", traindf['landmark_id'].nunique())

In [ ]:
MIN_CLASS = -1
MAX_CLASS = 500

traindf_s = traindf[(MIN_CLASS < traindf['landmark_id']) & (traindf['landmark_id'] <= MAX_CLASS)]

UNDERSAMPLING_THRESHOLD = 80

In [ ]:
!pip -q install -U datasets

from datasets import load_dataset
import os

dataset = load_dataset(
    "pemujo/GLDv2_Top_51_Categories",
    split="train",
    streaming=True
)

small_dataset = list(dataset.take(500))

os.makedirs("/content/landmark_small", exist_ok=True)

for i, item in enumerate(small_dataset):
    image = item["image"].convert("RGB")
    label = item["label"]
    image.save(f"/content/landmark_small/{i}_label{label}.jpg")

print("Images loaded:", len(small_dataset))
print("500 LANDMARK IMAGES READY")


In [ ]:
!pip -q install -U pyarrow datasets

In [ ]:
IMG_SIZE=120

SEED = 42
MIN_CLASS = -1
MAX_CLASS = 500

traindf_s = traindf[(MIN_CLASS < traindf['landmark_id']) & (traindf['landmark_id'] <= MAX_CLASS)]

UNDERSAMPLING_THRESHOLD = 80

traindf_s = (traindf_s.groupby('landmark_id', group_keys=False).
             apply(lambda x: x.sample(n=min(len(x), UNDERSAMPLING_THRESHOLD), random_state=SEED)))

traindf_s.reset_index(inplace=True, drop=True)

traindf_s['landmark_id'].value_counts()

N_DATA = len(traindf_s['landmark_id'])

clases = traindf_s['landmark_id'].unique()

N_CLASS = len(clases)

print('Sample data number, N_DATA: '+str(N_DATA))
print('\nNumber of classes in the sample, N_CLASS: '+str(N_CLASS))
print('\nMIN class of the sample: '+str(min(traindf_s['landmark_id'])))
print('\nMAX class of the sample: '+str(max(traindf_s['landmark_id'])))
print('\nRepetitions of elements by class in the sample after Undersampling:')
print(traindf_s['landmark_id'].value_counts())

In [ ]:
traindf_s['landmark_id'].value_counts().describe()

In [ ]:
import os

image_files = [
    os.path.join("/content/landmark_small", f)
    for f in os.listdir("/content/landmark_small")
    if f.endswith(".jpg")
]

N_DATA = len(image_files)

print("Images:", N_DATA)

In [ ]:
plt.figure(figsize=(25, 10))

for i in range(20):

    j = np.random.randint(0, N_DATA)

    img = plt.imread(image_files[j])

    plt.subplot(4, 5, i + 1)

    plt.imshow(img)

    plt.xticks([])

    plt.yticks([])

    plt.xlabel(
        'landmark_id: ' + str(traindf_s['landmark_id'][j]),
        fontweight="bold",
        fontsize=16
    )

    plt.imshow(img)


In [ ]:
X = []  # Images
y = []  # Classes

for i, item in enumerate(small_dataset):

    img = item["image"].convert("RGB").resize((128, 128))

    X.append(np.array(img))

    y.append(item["label"])

print('Variable types: \n')

print('X: ', type(X))

print('X elements: ', type(X[0]))

print('\ny: ', type(y))

print('y elements: ', type(y[0]))


# Label Encoding

LE = LabelEncoder()

LE.fit(y)

y_LE = LE.transform(y)


X = [n/255.0 for n in X]

X = np.array(X)

y_LE = np.array(y_LE)

y = np.array(y)


# Separating data for training, testing and validating

X_train, X_val, y_train, y_val = train_test_split(
    X,
    y_LE,
    test_size=0.10,
    random_state=SEED,
    shuffle=True
)

print("X shape:", X.shape)
print("y shape:", y.shape)
print("X_train:", X_train.shape)
print("X_val:", X_val.shape)

In [ ]:

X = []  # Images
y = []  # Classes
for i, item in enumerate(small_dataset):
    img = item["image"].convert("RGB").resize((128, 128))
    X.append(np.array(img))
    y.append(np.array(item["label"]))
print('Variable types: \n')
print('X: ', type(X))
print('X elements: ', type(X[0]))
print('\ny: ', type(y))
print('y elements: ', type(y[0]))
# Label Encoding
LE = LabelEncoder()
LE.fit(y)
y_LE = LE.transform(y)
X = [n/255 for n in X]
X = np.array(X)
y_LE = np.array(y_LE)
y = np.array(y)
# Separating data for training, testing and validating
X_train, X_val, y_train, y_val = train_test_split(X, y_LE, test_size=0.10, random_state=SEED, shuffle=True)

In [ ]:
# Separating data for training, testing and validating
X_train, X_val, y_train, y_val = train_test_split(X, y_LE, test_size = 0.10, random_state=SEED, shuffle=True)

X_train, X_test, y_train, y_test = train_test_split(X_train, y_train, test_size = 0.05, random_state=SEED, shuffle=True)

training_data = len(X_train)
training_data = len(X_train)
print('X-Images: ', len(X), 'and y labels: ', len(y_LE))
print('Training. Images in X_train: ', len(X_train), 'and labels in y_train: ', len(y_train))
print('Validation. Images in X_val: ', len(X_val), 'and labels in y_val: ', len(y_val))


In [ ]:

data_augmentation = tf.keras.Sequential([
    preprocessing.RandomZoom(height_factor=(-0.2, 0.2)),
    preprocessing.RandomContrast(0.4),
    preprocessing.RandomTranslation(height_factor=(-0.2, 0.2), width_factor=(-0.2, 0.2)),
    preprocessing.RandomRotation(factor=(-0.1, 0.1)),
])

plt.figure(figsize=(25,7))
j = 1
img_orig = X[j]
plt.subplot(2, 6, 1)
plt.xticks([])
plt.yticks([])
plt.xlabel('ORIGINAL. landmark_id: ' + str(y[j]), fontweight="bold", fontsize=12)
plt.imshow(img_orig)

for i in range(11):
    img_mod = data_augmentation(img_orig)
    plt.subplot(2, 6, i+2)
    plt.xticks([])
    plt.yticks([])
    plt.xlabel('Modification '+str(i+1), fontweight="bold", fontsize=12)
    plt.imshow(img_mod)

plt.show()

In [ ]:
# ---------------------------------------------------------
# 5. Oversampling Minority Classes
# ---------------------------------------------------------
start_time = time.time()
OVERSAMPLING_THRESHOLD = 0.2 * UNDERSAMPLING_THRESHOLD
i=0; n=0; k=0
for i in range(N_CLASS):
    cont=0
    for n in range(len(y_train)):
        if y_train[n] == i:
            cont = cont+1
    if cont > 0 and cont < OVERSAMPLING_THRESHOLD:
      img_class_i = X_train[y_train==i]

    if len(img_class_i) > 0:
      for k in range(int(OVERSAMPLING_THRESHOLD-cont)):
            X_train = np.concatenate((X_train,np.array([(img_class_i[np.random.randint(len(img_class_i))])])),axis = 0)
            y_train = np.append(y_train, i)
print('\nOversampling duration: %s seconds' % (time.time() - start_time))
n_data_train_def = len(X_train)
print('\nTraining data after oversampling: '+str(n_data_train_def)+' data')
print('\nThe training data has gone from: '+str(training_data)+' to:'+ str(n_data_train_def))


In [ ]:
from tensorflow.keras import regularizers
# ---------------------------------------------------------
# 6. CNN Hyperparameters Definition
# ---------------------------------------------------------
# CNN Hyperparameters
kernelSize = (3,3)
paddingType = 'same'
activationF = 'relu'
poolSize = (2,2)
stridesSize = (2,2)
dropoutRate = 0.5
batchSize=256
epochsSize= 1000
lr= 0.01
weightDecay=regularizers.l2(0.01)

In [ ]:
model = keras.Sequential([
    layers.Input(shape=(IMG_SIZE, IMG_SIZE, 3)),

    # DATA AUGMENTATION
    preprocessing.RandomZoom(height_factor=(-0.2, 0.2)),
    preprocessing.RandomContrast(0.4),
    preprocessing.RandomTranslation(height_factor=(-0.2, 0.2), width_factor=(-0.2, 0.2)),
    preprocessing.RandomRotation(factor=(-0.1, 0.1)),

    # MODEL BASIS ---- Feature Extraction
    # convolutional block 1
    layers.Conv2D(filters=32, kernel_size=kernelSize, strides=1, padding=paddingType, activation=activationF),
    layers.BatchNormalization(),
    layers.MaxPool2D(pool_size=poolSize, strides=stridesSize, padding=paddingType),

    # convolutional block 2
    layers.Conv2D(filters=64, kernel_size=kernelSize, strides=1, padding=paddingType, activation=activationF),
    layers.BatchNormalization(),
    layers.MaxPool2D(pool_size=poolSize, strides=stridesSize, padding=paddingType),

    # convolutional block 3
    layers.Conv2D(filters=128, kernel_size=kernelSize, strides=1, padding=paddingType, activation=activationF),
    layers.BatchNormalization(),
    layers.MaxPool2D(pool_size=poolSize, strides=stridesSize, padding=paddingType),

    # convolutional block 4
    layers.Conv2D(filters=256, kernel_size=kernelSize, strides=1, padding=paddingType, activation=activationF),
    layers.BatchNormalization(),
    layers.MaxPool2D(pool_size=poolSize, strides=stridesSize, padding=paddingType),

    # convolutional block 5
    layers.Conv2D(filters=512, kernel_size=kernelSize, strides=1, padding=paddingType, activation=activationF),
    layers.BatchNormalization(),
    layers.MaxPool2D(pool_size=poolSize, strides=stridesSize, padding=paddingType),

    # MODEL HEAD ---- Classification
    layers.Flatten(),
    layers.Dense(units=512, activation=activationF, kernel_regularizer=weightDecay),
    layers.Dropout(rate=dropoutRate, seed=SEED),
    layers.BatchNormalization(),

    # Output Layer
    layers.Dense(units=N_CLASS, activation='softmax'),
])

# OPTIMIZERS
# The lower lr, the less sudden changes there are in metrics such as accuracy
opt1 = tf.keras.optimizers.RMSprop(learning_rate=lr, rho=0.9, momentum=0.5, epsilon=1e-07)
opt2 = tf.keras.optimizers.Adam(learning_rate=lr, beta_1=0.9, beta_2=0.999, epsilon=1e-07)
opt3 = tf.keras.optimizers.SGD(learning_rate=lr, momentum=0.9, nesterov=False)

# Model compilation
model.compile(
    optimizer = opt3,
    loss = 'sparse_categorical_crossentropy',
    metrics = ['sparse_categorical_accuracy']
)

# Callback 1
early_stopping = EarlyStopping(
    monitor = "val_loss",
    mode = "auto",
    min_delta = 0.0001,
    patience = 15,
    restore_best_weights = True)

# Callback 2
reduce_lr = ReduceLROnPlateau(
    monitor='val_loss', factor=0.7,
    patience=40, cooldown=1, min_lr=0.0005,
    min_delta=0.001, verbose=1)

# Callback 3
checkpoint = ModelCheckpoint(
    'best-weights.weights.h5', monitor='val_loss',
    save_best_only=True, save_weights_only=True)

from sklearn.utils import class_weight
classWeights = class_weight.compute_class_weight(class_weight='balanced',classes=np.unique(y_train),y=y_train)
train_classWeights = dict(enumerate(classWeights))

# Calculation of weights of each class of data for sample training
_, freq = np.unique(y_train, return_counts=True)
max_freq = np.max(freq)
print('Highest frequency: '+str(np.max(freq)))
start_time = time.time()


# Model
history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    class_weight=train_classWeights,
    shuffle=True,
    batch_size=batchSize,
    epochs=epochsSize,
    callbacks=[early_stopping, reduce_lr, checkpoint],
    verbose=1
)



In [ ]:
history_df = pd.DataFrame(history.history)

print('\nbatchSize = '+str(batchSize))

plt.figure(figsize=(15,5))

plt.subplot(1,2,1)
plt.plot(history.history['loss'], label='Entrenamiento')
plt.plot(history.history['val_loss'], label='Validación')
plt.legend()
plt.title('Loss and Validation Loss')
plt.xlabel('Epoch')
print("Minimum Validation Loss: {:0.4f} in epoch {:0.0f} ".format(history_df['val_loss'].min(), history_df['val_loss'].idxmin()))

plt.subplot(1,2,2)
plt.plot(history.history['sparse_categorical_accuracy'], label='Training')
plt.plot(history.history['val_sparse_categorical_accuracy'], label='Validation')
plt.legend()
plt.title('Accuracy and Validation Accuracy')
plt.xlabel('Epoch')
print("Maximum Validation Accuracy: {:0.4f} in epoch {:0.0f} ".format(history_df['val_sparse_categorical_accuracy'].max(), history_df['val_sparse_categorical_accuracy'].idxmax()))

print("\nEvaluation of the model with training data")
score = model.evaluate(X_train, y_train)
print("Test loss, Test accuracy:", score[0], score[1])

print("\nModel evaluation with validation data")
score = model.evaluate(X_val, y_val)
print("Test loss, Test accuracy:", score[0], score[1])

plt.show()

In [ ]:
model.summary()


In [ ]:
plt.figure(figsize=(25,7))

for i in range (12):
  img=X_test[i]
  plt.subplot(2, 6, i+1)
  plt.xticks([])
  plt.yticks([])
  plt.xlabel('Class: determined')
  plt.imshow(img)

In [ ]:
predict=model.predict(X_test,verbose=1)

In [ ]:
y_pred=[]

In [ ]:
img_class_df = pd.DataFrame({
    'image': image_files,
    'label': [item["label"] for item in small_dataset]
})

In [ ]:
col = 6
plt.figure(figsize=(18,6))

for i in range(len(img_class_df)):
  if i<(col-1):
    img_class_path = img_class_df.iloc[i,0]
    img_class = img_read_resize(img_class_path)

    prediction = model.predict(np.expand_dims(img_class, axis=0), verbose=0)
    pred_class = np.argmax(prediction[0])
    confidence = np.max(prediction[0]) * 100

    actual_class = img_class_df.iloc[i,1]

    plt.subplot(1,col,i+2)
    plt.xticks([])
    plt.yticks([])
    plt.title('Actual: '+str(actual_class)+
              '\nPredicted: '+str(pred_class)+
              '\nConfidence: '+str(round(confidence,2))+'%', fontsize=9)
    plt.imshow(img_class)
  else:
    break

plt.tight_layout()
plt.show()

In [ ]:
prediction = model.predict(np.expand_dims(img_class, axis=0), verbose=0)

print("Actual class:", actual_class)
print("Predicted class:", np.argmax(prediction[0]))
print("Confidence:", np.max(prediction[0]) * 100)
print("All probabilities:", prediction[0])

In [ ]:
N_CLASS = len(LE.classes_)
print("N_CLASS =", N_CLASS)

In [ ]:
print("Number of classes:", len(LE.classes_))
print("Model output classes:", model.output_shape[-1])

In [ ]:
selected_landmarks = [138982, 126637, 20409, 83144, 113209]

small_df = (
    traindf[traindf['landmark_id'].isin(selected_landmarks)]
    .groupby('landmark_id')
    .head(20)
    .reset_index(drop=True)
)

print("Total selected images:", len(small_df))
print(small_df['landmark_id'].value_counts())

In [ ]:
import os

small_df['img_path'] = small_df['id'].apply(
    lambda x: os.path.join(
        'train',
        x[0],
        x[1],
        x[2],
        x + '.jpg'
    )
)

print(small_df[['id', 'landmark_id', 'img_path']].head())

In [ ]:
!pip install -q kaggle

In [ ]:
print(traindf.shape)
print(traindf.columns.tolist())
traindf.head()

In [ ]:
# Google Landmark Recognition 2021 - SMALL DATASET
# Downloads only 500 images instead of the ~100 GB competition dataset

!pip -q install kagglehub pandas requests tqdm

import os
import random
import requests
import pandas as pd
from tqdm.auto import tqdm
import kagglehub

# -----------------------------
# 1. Download ONLY train.csv
# -----------------------------
print("Downloading train.csv...")

metadata_path = kagglehub.competition_download(
    "landmark-recognition-2021",
    path="train.csv"
)

print("Metadata:", metadata_path)

# Find the CSV
if os.path.isdir(metadata_path):
    train_csv = os.path.join(metadata_path, "train.csv")
else:
    train_csv = metadata_path

df = pd.read_csv(train_csv)

print("Total training images:", len(df))
print("Columns:", list(df.columns))

# -----------------------------
# 2. Select 500 images
# -----------------------------
N_IMAGES = 500

# Take images from different landmark classes
sample_df = (
    df.groupby("landmark_id", group_keys=False)
      .apply(lambda x: x.sample(min(len(x), 5), random_state=42))
      .head(N_IMAGES)
      .reset_index(drop=True)
)

print("Selected images:", len(sample_df))
print("Different landmarks:", sample_df["landmark_id"].nunique())

# -----------------------------
# 3. Download selected images
# -----------------------------
BASE_URL = "https://lh3.googleusercontent.com/"

output_dir = "/content/landmark_small"
os.makedirs(output_dir, exist_ok=True)

success = 0
failed = 0

for _, row in tqdm(sample_df.iterrows(), total=len(sample_df)):

    image_id = row["id"]

    # GLDv2 image folder structure:
    # abcdef... -> a/b/c/abcdef....jpg
    image_url = (
        BASE_URL
        + image_id[0]
        + "/"
        + image_id[1]
        + "/"
        + image_id[2]
        + "/"
        + image_id
        + ".jpg"
    )

    save_path = os.path.join(output_dir, image_id + ".jpg")

    if os.path.exists(save_path):
        success += 1
        continue

    try:
        r = requests.get(image_url, timeout=15)

        if r.status_code == 200 and len(r.content) > 1000:
            with open(save_path, "wb") as f:
                f.write(r.content)
            success += 1
        else:
            failed += 1

    except:
        failed += 1

print("\n==============================")
print("SMALL DATASET READY")
print("==============================")
print("Images downloaded:", success)
print("Images failed:", failed)
print("Dataset folder:", output_dir)

In [ ]:
import pandas as pd
import os

train_csv = "/root/.cache/kagglehub/competitions/landmark-recognition-2021/train.csv"

# Read only the first 500 rows
df = pd.read_csv(train_csv, nrows=500)

print("Images selected:", len(df))
print("Columns:", list(df.columns))
print(df.head())

# Create folder
os.makedirs("/content/landmark_small", exist_ok=True)

print("\nReady to download 500 images.")

In [ ]:
import os
import requests
from tqdm.auto import tqdm

output_dir = "/content/landmark_small"
os.makedirs(output_dir, exist_ok=True)

success = 0
failed = 0

for _, row in tqdm(df.iterrows(), total=len(df)):

    image_id = str(row["id"])

    # Google Landmark Recognition 2021 image URL
    url = (
        "https://lh3.googleusercontent.com/"
        + image_id[0] + "/"
        + image_id[1] + "/"
        + image_id[2] + "/"
        + image_id
        + "=w512"
    )

    save_path = os.path.join(output_dir, image_id + ".jpg")

    if os.path.exists(save_path):
        success += 1
        continue

    try:
        response = requests.get(url, timeout=20)

        if response.status_code == 200 and len(response.content) > 1000:
            with open(save_path, "wb") as f:
                f.write(response.content)
            success += 1
        else:
            failed += 1

    except Exception:
        failed += 1

print("\n==============================")
print("DOWNLOAD COMPLETE")
print("==============================")
print("Successfully downloaded:", success)
print("Failed:", failed)
print("Folder:", output_dir)


In [ ]:
!pip -q install -U datasets

from datasets import load_dataset
import os

print("Loading small Google Landmark dataset...")

dataset = load_dataset(
    "pemujo/GLDv2_Top_51_Categories",
    split="train",
    streaming=True
)

# Take only 500 images
small_dataset = list(dataset.take(500))

print("Images loaded:", len(small_dataset))

# Create folder
os.makedirs("/content/landmark_small", exist_ok=True)

# Save images and labels
for i, item in enumerate(small_dataset):
    image = item["image"]
    label = item["label"]
    landmark_id = item["landmark_id"]
    category = item["category"]

    image.save(
        f"/content/landmark_small/{i}_label{label}.jpg"
    )

print("================================")
print("500 LANDMARK IMAGES READY")
print("================================")
print("Folder: /content/landmark_small")

In [ ]:
import os
import matplotlib.pyplot as plt
from PIL import Image

folder = "/content/landmark_small"

files = [
    f for f in os.listdir(folder)
    if f.endswith(".jpg")
]

print("Total images:", len(files))

# Show 9 sample images
plt.figure(figsize=(12, 12))

for i, file in enumerate(files[:9]):
    img = Image.open(os.path.join(folder, file))

    plt.subplot(3, 3, i + 1)
    plt.imshow(img)
    plt.title(file)
    plt.axis("off")

plt.tight_layout()
plt.show()


In [ ]:
N_CLASS = len(LE.classes_)
print("N_CLASS =", N_CLASS)